# Lakeflow Designer Workshop: Consulting Billing & Utilisation

A hands-on tutorial built around a fictional professional services firm, **Northgate Advisory LLP**. Attendees turn raw timesheet exports, expense claims, rate cards and Statement of Work PDFs into governed billing tables. Along the way they use Lakeflow Designer's no-code operators, Genie Code and job scheduling, and compare the result with the same logic written in a notebook. AI document parsing of the Statement of Work PDFs is in the separate [3. Extension - PDF Parsing](#notebook-102205435070834) notebook, which leads into the AI workshops after the break.

## The scenario

Northgate Advisory is a 60-person consulting firm with five practices: Audit, Technology Consulting, Deal Advisory, Risk Consulting and Tax. Every month the finance team has to answer the same questions:
* How much work in progress (WIP) has each engagement built up, at standard rates and at the rates agreed with the client?
* What is the **rate realisation** (agreed rate ÷ standard rate) and the margin on each engagement?
* Which engagements are about to go over their fee cap or fixed fee?
* Who is over or under their utilisation target?
* Are there timesheet entries that need a closer look: weekend working, entries over 12 hours, or time booked to an engagement the person isn't assigned to?

The structured master data is already held in Unity Catalog. Timesheets and expenses arrive as file exports, and the commercial terms only exist in signed **Statement of Work (SOW) PDFs**.

## Data model

| Asset | Type | Location | Description |
| --- | --- | --- | --- |
| `employees` | Delta table | schema | 60 staff: grade, practice, office, line manager (`manager_id`), hire date, utilisation target |
| `clients` | Delta table | schema | 12 clients: industry, country, billing currency, tier, relationship partner |
| `engagements` | Delta table | schema | 16 client engagements plus 4 internal time codes: billing type, engagement partner and manager, dates, SOW reference |
| `engagement_assignments` | Delta table | schema | Who is staffed on which engagement, their role and allocation % |
| `timesheets/timesheets_2026_MM.csv` | CSV files | volume | Monthly timesheet-system exports (Aug-Oct 2026). Contains messy values on purpose |
| `expenses/expenses_2026_MM.json` | JSON lines | volume | Expense claims: category, merchant, amount, currency, rechargeable flag |
| `reference/grade_rate_card.csv` | CSV file | volume | Standard and cost day rates for each grade |
| `reference/fx_rates_monthly.csv` | CSV file | volume | Monthly FX rates to GBP |
| `contracts/*.pdf` | PDF files | volume | 16 Statements of Work (fee, agreed day rates, payment terms, PO) plus 1 change request |

A working day is **7.5 hours**. Day rates are in GBP.

**Problems deliberately seeded in the timesheet files** (for attendees to find): dates stored as `dd/MM/yyyy` text, engagement codes with trailing spaces or in lowercase, lowercase `y`/`n` billable flags, 12 weekend entries, 8 entries over 12 hours, and 10 entries booked by people not assigned to the engagement. Three engagements (ENG-26-003, ENG-26-007, ENG-26-010) are built to run over their fee.

## Agenda

| Part | Topic | Format |
| --- | --- | --- |
| 0 | Setup: load the sample data (facilitator, once, in the [1. Workshop Data Setup](#notebook-3120290690157533) notebook) | Run notebook |
| 1 | Lakeflow Designer: no-code timesheet WIP flow | Hands-on |
| 2 | Genie Code in Designer | Hands-on |
| 3 | Schedule the flow as a job | Hands-on |
| 4 | Contrast: the same pipeline in a notebook | Walkthrough |
| Extension | Lakeflow Designer: parse SOW PDFs with `ai_parse_document` and `ai_extract` ([3. Extension - PDF Parsing](#notebook-102205435070834) notebook), leading into the AI sessions after the break | Facilitator demo |

## Before you start

The facilitator loads the ready-made sample data once with the [1. Workshop Data Setup](#notebook-3120290690157533) notebook in this folder. It creates the four Delta tables, the CSV/JSON exports, the Statement of Work PDFs and an answer key for the AI extraction exercise.

Attendees don't need to run it. Set the widgets below to the same catalog, schema and volume that the setup notebook used, then run the next cell.

In [0]:
dbutils.widgets.text("catalog", "praju_stable_2_catalog", "Catalog")
dbutils.widgets.text("schema", "consulting_billing", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

FQ_SCHEMA = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Fail fast if the setup notebook hasn't been run for this catalog/schema
missing = [t for t in ["employees", "clients", "engagements", "engagement_assignments"]
           if not spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{t}")]
if missing:
    raise RuntimeError(f"Missing tables {missing}: run the '1. Workshop Data Setup' notebook first.")

print(f"Schema : {CATALOG}.{SCHEMA}")
print(f"Volume : {VOLUME_PATH}")

## Part 1: Lakeflow Designer, no-code transformations (timesheets to WIP)

**Goal:** turn three raw monthly timesheet exports into a clean `engagement_wip_monthly` table that shows hours, WIP at standard rates and cost for each engagement and month, without writing any code.

**Two ways to build each step.** Every step below has a manual route (drag and configure operators) and a **Genie Code prompt** that does the same thing in natural language. Pick one:
* **Manual:** builds confidence with the operators. Recommended for first-time users.
* **Genie Code:** open the Genie Code panel inside the visual data prep and paste the prompt for that step. Review the operators it adds, and check the preview, before accepting.
* **Mixed (recommended for the workshop):** build steps 2-4 by hand, then use Genie Code for steps 5-8 and compare.

After each Genie Code step, check that the preview row count and the columns are what you expected. If they aren't, reply with a correction (for example *"use a left join instead"*) rather than starting over.

**1. Create the visual data prep**
* In the left navigation click **New > Visual data prep**. Rename the tab to `<initials>_timesheet_wip`.
* Open the **Genie Code** panel from the right-hand side of the canvas. You will use it in each step below.

**2. Source: the whole timesheets folder**
* Add a **Source** operator and point it at the volume folder `/Volumes/<catalog>/consulting_billing/raw_files/timesheets/`. Designer combines all three CSV files into one table.
* Preview the data and look for the problems: `work_date` is `dd/MM/yyyy` text, some `engagement_code` values have trailing spaces or are in lowercase, and `billable` contains both `Y` and `y`.

> **Genie Code prompt:** *Add a source that reads all the CSV files in the volume folder /Volumes/<catalog>/consulting_billing/raw_files/timesheets/ with a header row. Then tell me which columns have data quality issues.*

**3. Prepare: clean the columns** (one **Prepare** operator, several actions)

| Action | Column | Setting |
| --- | --- | --- |
| Trim | `engagement_code`, `billable` | Both ends |
| Text case | `engagement_code`, `billable` | Upper |
| Parse date | `work_date` | Format `dd/MM/yyyy` |
| Change type | `hours` | Double |
| Formula | new column `days` | `hours / 7.5` |

> **Genie Code prompt:** *Add a Prepare step that trims and upper-cases engagement_code and billable, parses work_date as a date in dd/MM/yyyy format, converts hours to a double, and adds a column days = hours / 7.5.*

Tip: inside a **Formula** action you can also type a description instead of SQL, for example *"hours divided by 7.5"*. Designer generates the expression for you.

**4. Filter: approved, billable time only**
* Add a **Filter** with `approval_status = Approved` AND `billable = Y`.

> **Genie Code prompt:** *Keep only timesheet rows where approval_status is Approved and billable is Y.*

**5. Join the master data**

| Join to | Source | Key | Type |
| --- | --- | --- | --- |
| Employees | table `employees` | `employee_id` | Inner |
| Rate card | file `raw_files/reference/grade_rate_card.csv` | `grade` | Inner |
| Engagements | table `engagements` | `engagement_code` | Inner |
| Clients | table `clients` | `client_id` | Left |

> **Genie Code prompt:** *Join the timesheets to the employees table on employee_id, to the rate card file /Volumes/<catalog>/consulting_billing/raw_files/reference/grade_rate_card.csv on grade, and to the engagements table on engagement_code. Then left join clients on client_id. Use the tables in <catalog>.consulting_billing.*

If you skip the Trim/Text case step, the engagement join silently drops about 6% of rows. Compare the row counts before and after.

**6. Prepare: calculate the values** (Formula actions)
* `billing_month` = `date_trunc('MONTH', work_date)`
* `wip_standard_gbp` = `days * standard_day_rate_gbp`
* `cost_gbp` = `days * cost_day_rate_gbp`

> **Genie Code prompt:** *Add columns billing_month (the first day of the month of work_date), wip_standard_gbp = days x standard_day_rate_gbp, and cost_gbp = days x cost_day_rate_gbp.*

**7. Aggregate**
* Group by `engagement_code`, `engagement_name`, `client_name`, `billing_type`, `billing_month`.
* Sum `hours`, `wip_standard_gbp` and `cost_gbp`.

> **Genie Code prompt:** *Aggregate by engagement_code, engagement_name, client_name, billing_type and billing_month. Sum hours as billable_hours, and sum wip_standard_gbp and cost_gbp.*

**8. Output**
* **Output** type *Table*, name `<initials>_engagement_wip_monthly`, write mode *Overwrite*. Click **Run**.

> **Genie Code prompt:** *Write the result to a table called <initials>_engagement_wip_monthly in <catalog>.consulting_billing, overwriting it on each run.*

**Challenge: the whole flow in one prompt.** In a new, empty visual data prep, paste the prompt below and compare the graph Genie Code builds with your step-by-step version. Discuss which one is easier to review.

> *Using the timesheet CSVs in /Volumes/<catalog>/consulting_billing/raw_files/timesheets/, build a flow that: trims and upper-cases engagement_code and billable; parses work_date from dd/MM/yyyy; converts hours to a double and adds days = hours / 7.5; keeps Approved, billable (Y) rows; joins employees on employee_id, the rate card CSV in raw_files/reference on grade, and engagements on engagement_code, then left joins clients on client_id; calculates billing_month, wip_standard_gbp = days x standard_day_rate_gbp and cost_gbp = days x cost_day_rate_gbp; aggregates by engagement_code, engagement_name, client_name, billing_type and billing_month (sum hours as billable_hours, plus wip_standard_gbp and cost_gbp); and writes the result to <initials>_engagement_wip_monthly.*

**Ask Genie Code to explain:** select any operator and ask *"What does this step do, and how many rows does it remove?"* This is a good way to review a flow someone else has built.

**Checkpoint:** compare your output with the answer-key table `nb_engagement_monthly_billing` created in Part 4. The `billable_hours` and `wip_standard_gbp` values should match exactly.

**Extra:** open the table of contents and view the generated code. Every operator is backed by SQL/PySpark, and the file is saved as `<name>.designer.ipynb`.

## Part 2: Genie Code in Lakeflow Designer

Genie Code can add, change and explain operators from natural language. Try these three ways of using it.

**A. AI operator search:** in the Operators pane, search for *"total by month"* or *"remove duplicates"* and look at the **AI suggested** operators.

**B. Edit an operator by its description:** every operator shows an AI-generated description. Edit the description of your Filter, for example to *"keep approved billable time, and submitted time from the last week"*, and watch the configuration update.

**C. Build steps with the Genie Code panel.** Open Genie Code in your Part 1 prep and try these prompts. For each one, review the operators it proposes before accepting them.

| # | Prompt | What a correct answer finds |
| --- | --- | --- |
| 1 | *Flag timesheet entries that fall on a weekend or are over 12 hours, and write them to an exceptions table.* | 12 weekend entries and 8 entries over 12 hours |
| 2 | *Find timesheet entries where the employee isn't assigned to that engagement in engagement_assignments.* | 10 entries |
| 3 | *Calculate monthly utilisation per employee: billable hours divided by available hours (7.5 x working days), and compare it with annual_utilisation_target.* | Partners well below 0.40; some staff on the bench below target |
| 4 | *Load the expenses JSON files, convert amounts to GBP using fx_rates_monthly, and total rechargeable expenses per engagement.* | EUR on the Thistle Insurance engagement and USD on Northstar Telecom; no rechargeable expenses on "included in fees" SOWs |
| 5 | *Compare WIP at contracted rates to date with the fee in contract_terms, and flag engagements over 90% of fee.* (Uses the curated `contract_terms` and `contracted_rates` tables; the [3. Extension - PDF Parsing](#notebook-102205435070834) notebook shows how they're extracted from the PDFs.) | ENG-26-003, ENG-26-007 and ENG-26-010 are over fee |
| 6 | *Add each employee's line manager name using manager_id.* | A self-join of `employees` |

**Good practice:** ask Genie Code to explain a step (*"What does this join do?"*) and check the preview row counts after each change.

## Part 3: Job scheduling (hands-on)

**1. Schedule the prep**
* In `<initials>_timesheet_wip` click **Schedule > Add schedule**: daily at 07:00, time zone Europe/London.
* Under notifications, add your email address for **failure** alerts.

**2. Run it and inspect the run**
* From the schedule list, open the kebab menu and click **Run now**, then **View in Jobs**.
* The run shows **each operator as a separate node** in the task graph. Click a node to see its output, the same way you would on the canvas. Run history gives an audit trail of every refresh.

**3. Simulate new data arriving**
* The facilitator runs the *Drop November Timesheet File* cell at the end of the [1. Workshop Data Setup](#notebook-3120290690157533) notebook. It writes `timesheets_2026_11.csv` into the same folder.
* Click **Run now** again. November rows now appear in `<initials>_engagement_wip_monthly`, with no change to the flow, because the Source reads the whole folder.

**4. Make it a pipeline of tasks** (optional, facilitator demo)
* In **Jobs & Pipelines**, create a job with two tasks of type **Visual data prep**: the demo prep `sow_extraction_demo` from the [3. Extension - PDF Parsing](#notebook-102205435070834) notebook, then `<initials>_timesheet_wip` (which depends on the first). Attendees can do this with their own `<initials>_timesheet_wip` as the only task.
* Change the trigger to **File arrival** on the `timesheets/` folder, so the job runs when a new export lands rather than on a timer.

**5. Clean up:** pause or delete your schedule after the workshop.

## Part 4: Contrast, the same pipeline in a notebook

The cells below rebuild Part 1 in SQL, with one PySpark example, and write `nb_*` tables. These also act as the **answer key** for the Designer exercises. *SQL: Monthly Engagement Billing* also values WIP at contracted rates, using `nb_contracted_rates`. That table is created by the SQL cells in the [3. Extension - PDF Parsing](#notebook-102205435070834) notebook, along with the fee burn and the AI accuracy check.

| | Lakeflow Designer | SQL notebook | Python (PySpark) notebook |
| --- | --- | --- | --- |
| Best for | Analysts and auditors; repeatable data prep | Analysts and engineers comfortable with SQL | Engineers; complex logic, libraries, tests |
| How it's built | Drag-and-drop operators, Genie Code, a preview at every step | SQL statements | DataFrame API and Python functions |
| Transparency | Visual graph; generated code viewable | The code is the documentation | The code is the documentation |
| Reuse | Custom and shared operators, parameters | Views, SQL functions | Python modules, unit tests |
| Version control | Git (`.designer.ipynb`), visual version diff | Git | Git |
| Scheduling | Schedule button, or a Visual data prep job task | Notebook job task | Notebook job task |
| Governance | Unity Catalog, same lineage | Unity Catalog, same lineage | Unity Catalog, same lineage |

All three run on the same compute, read the same governed tables and appear in the same lineage graph. The choice is about **who maintains it**, not about what the platform can do.

In [0]:
# Point the session at the workshop schema and expose the raw files as temporary views,
# so the SQL cells below can read them by name.
spark.sql(f"USE {FQ_SCHEMA}")

spark.sql(f"""
  CREATE OR REPLACE TEMP VIEW raw_timesheets AS
  SELECT *, _metadata.file_name AS source_file
  FROM read_files('{VOLUME_PATH}/timesheets/', format => 'csv', header => true, inferSchema => false)
""")
spark.sql(f"""
  CREATE OR REPLACE TEMP VIEW raw_rate_card AS
  SELECT * FROM read_files('{VOLUME_PATH}/reference/grade_rate_card.csv', format => 'csv', header => true)
""")
display(spark.sql("SELECT source_file, COUNT(*) AS rows FROM raw_timesheets GROUP BY source_file ORDER BY source_file"))

In [0]:
%sql
-- Part 1, steps 2-3 in SQL: clean the raw timesheet export
CREATE OR REPLACE TABLE nb_timesheets_clean AS
SELECT
  timesheet_id,
  employee_id,
  upper(trim(engagement_code))           AS engagement_code,
  to_date(work_date, 'dd/MM/yyyy')       AS work_date,
  CAST(hours AS DOUBLE)                  AS hours,
  CAST(hours AS DOUBLE) / 7.5            AS days,
  activity,
  upper(trim(billable)) = 'Y'            AS is_billable,
  approval_status,
  comments,
  source_file
FROM raw_timesheets;

SELECT approval_status, is_billable, COUNT(*) AS entries, SUM(hours) AS hours
FROM nb_timesheets_clean
GROUP BY ALL
ORDER BY ALL;

In [0]:
# The same cleaning step in PySpark, for comparison
from pyspark.sql import functions as F

ts_py = (
    spark.table("raw_timesheets")
    .select(
        "timesheet_id",
        "employee_id",
        F.upper(F.trim("engagement_code")).alias("engagement_code"),
        F.to_date("work_date", "dd/MM/yyyy").alias("work_date"),
        F.col("hours").cast("double").alias("hours"),
        (F.col("hours").cast("double") / 7.5).alias("days"),
        "activity",
        (F.upper(F.trim("billable")) == "Y").alias("is_billable"),
        "approval_status",
        "comments",
        "source_file",
    )
)

# Should be 0 rows different from the SQL version
print("Rows different from the SQL version:", ts_py.exceptAll(spark.table("nb_timesheets_clean")).count())

In [0]:
%sql
-- Part 1 plus contracted rates (nb_contracted_rates comes from the Extension - PDF Parsing notebook): monthly WIP at standard and contracted rates, cost, margin and realisation
CREATE OR REPLACE TABLE nb_engagement_monthly_billing AS
SELECT
  t.engagement_code,
  e.engagement_name,
  c.client_name,
  e.billing_type,
  CAST(date_trunc('MONTH', t.work_date) AS DATE)                    AS billing_month,
  SUM(t.hours)                                                      AS billable_hours,
  ROUND(SUM(t.days * rc.standard_day_rate_gbp), 2)                  AS wip_standard_gbp,
  ROUND(SUM(t.days * cr.agreed_day_rate_gbp), 2)                    AS wip_contract_gbp,
  ROUND(SUM(t.days * rc.cost_day_rate_gbp), 2)                      AS cost_gbp,
  ROUND(SUM(t.days * cr.agreed_day_rate_gbp) - SUM(t.days * rc.cost_day_rate_gbp), 2) AS margin_gbp,
  ROUND(SUM(t.days * cr.agreed_day_rate_gbp) / SUM(t.days * rc.standard_day_rate_gbp), 3) AS rate_realisation,
  SUM(CASE WHEN cr.agreed_day_rate_gbp IS NULL THEN t.hours ELSE 0 END) AS hours_without_contract_rate
FROM nb_timesheets_clean t
JOIN employees   emp ON t.employee_id = emp.employee_id
JOIN engagements e   ON t.engagement_code = e.engagement_code
JOIN clients     c   ON e.client_id = c.client_id
JOIN raw_rate_card rc ON emp.grade = rc.grade
LEFT JOIN nb_contracted_rates cr ON t.engagement_code = cr.engagement_code AND emp.grade = cr.grade
WHERE t.approval_status = 'Approved'
  AND t.is_billable
  AND e.engagement_type = 'Client'
GROUP BY ALL;

SELECT * FROM nb_engagement_monthly_billing ORDER BY engagement_code, billing_month;

## Facilitator answer key (Aug-Oct 2026 data, before the November file is added)

| Check | Expected value |
| --- | --- |
| Raw timesheet rows (3 files) | 7,101 |
| Raw rows with a dirty `engagement_code` (spaces or lowercase) | 6.1% |
| Approved, billable client hours | 13,063 |
| WIP at standard rates | GBP 1,666,377 |
| Rows in `engagement_wip_monthly` (engagement x month) | 40 |
| Weekend entries / entries over 12 hours / not-assigned entries | 12 / 8 / 10 |

The PDF extraction checks (contracted rates, fee caps, fee burn) are in the answer key at the end of the [3. Extension - PDF Parsing](#notebook-102205435070834) notebook. Running *Drop November Timesheet File* in the [1. Workshop Data Setup](#notebook-3120290690157533) notebook adds a fourth CSV, so the hours and WIP totals above will increase.